<a href="https://colab.research.google.com/github/takatakamanbou/AdvML/blob/2025/AdvML2025_ex09notebookA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AdvML ex09notebookA

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/AdvML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?AdvML)




板書や口頭で補足する前提なので，この notebook だけでは説明が不完全です．


In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

from scipy.stats import multivariate_normal
from sklearn.mixture import GaussianMixture

# NumPy の 疑似乱数生成器（rng = random number generator）
from numpy.random import default_rng
rng = default_rng() # 疑似乱数生成器を初期化

---
## 統計的識別入門 (2)
---


---
### 復習

識別問題の特徴量を $\pmb{x}$，クラスを $y$ で表す．クラス事後確率 $p(y|\pmb{x})$ を推定したい．これを直接モデル化するのが **識別モデル**．

$$
p(y|\pmb{x}) = \frac{p(\pmb{x}|y)p(y)}{p(\pmb{x})} \propto p(\pmb{x}|y)p(y)
$$

より $p(\pmb{x}|y)$ と $p(y)$ をモデル化するのが **生成モデル**．


---
### 生成モデルを用いた識別の例1: 2次元データ

2次元のデータを生成モデルを用いて識別する実験をやってみよう．

まずは学習データの準備．

In [ ]:
# データの読み込み
K = 3 # クラス数
D = 2 # 特徴次元数
df = pd.read_csv('https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/2dim3class.csv')

X = df.drop(columns='label').to_numpy()
y = df['label'].to_numpy()

df

In [ ]:
# グラフを描く
fig, ax = plt.subplots()
for k in range(K):
    ax.scatter(X[y==k, 0], X[y==k, 1], label=f'class{k}', s=5)
xmin, xmax = -5, 5
ymin, ymax = -5, 5
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')
ax.legend()
plt.show()

事前確率は，学習データの出現頻度で推定することにする．

In [ ]:
# 事前確率を推定
py = np.empty(K)
for k in range(K):
    py[k] = np.sum(y==k) / len(y)
print('### 事前確率')
print('p(y) = ', py)


クラスごとの分布には2次元正規分布を当てはめる．

In [ ]:
# p(x|y) を正規分布でモデル化．パラメータ（平均と共分散行列）を推定
mu = np.empty((K, D))
cov = np.empty((K, D, D))
for k in range(K):
    XX = X[y==k, :]
    mu[k] = np.mean(XX, axis=0)
    cov[k] = (XX - mu[k]).T @ (XX - mu[k]) / XX.shape[0]
    print(f'### class {k} の正規分布のパラメータ')
    print(mu[k])
    print(cov[k])
    print()

次のコードを実行すると，得られたモデルを用いて，2次元平面上の各点での事後確率を求め，それを可視化することができる．

In [ ]:
# グラフ描画用のグリッドデータの作成
x_mesh, y_mesh = np.mgrid[xmin:xmax:0.02, ymin:ymax:0.02]
X_mesh = np.dstack((x_mesh, y_mesh))

# 事後確率の推定
p = np.empty((K, X_mesh.shape[0]*X_mesh.shape[1]))
for k in range(K):
    p[k, :] = py[k] * multivariate_normal.pdf(X_mesh, mean=mu[k], cov=cov[k]).reshape(-1)
p /= np.sum(p, axis=0)
pp = p.reshape((K, X_mesh.shape[0], X_mesh.shape[1]))

# グラフ
fig = plt.figure(figsize=(9, 6))

# Gaussian を当てはめた結果
ax0 = fig.add_subplot(121)
for k in range(K):
    ax0.scatter(X[y==k, 0], X[y==k, 1], s=5)
    ax0.contour(x_mesh, y_mesh, multivariate_normal.pdf(X_mesh, mean=mu[k], cov=cov[k]))
ax0.set_xlim(xmin, xmax)
ax0.set_ylim(ymin, ymax)
ax0.set_aspect('equal')

# 事後確率の可視化
cmap = ['Blues', 'Oranges', 'Greens']
ax1 = fig.add_subplot(122)
for k in range(K):
    ax1.scatter(X[y==k, 0], X[y==k, 1], s=5)
    ax1.contourf(x_mesh, y_mesh, pp[k], levels=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], cmap=cmap[k], alpha=0.3)
ax1.set_xlim(xmin, xmax)
ax1.set_ylim(ymin, ymax)
ax1.set_aspect('equal')

plt.show()

---
### 多次元正規分布についての補足

$$
p(\pmb{x}; \pmb{\mu}, \Sigma) = \frac{1}{\sqrt{(2\pi)^D|\Sigma|}} \exp{ \left( -\frac{1}{2} (\pmb{x}-\pmb{\mu})^{\top}\Sigma^{-1}(\pmb{x}-\pmb{\mu}) \right) }
$$


---
### 生成モデルを用いた識別の例2: 手書き数字識別

手書き数字のデータを生成モデルを用いて識別する実験をやってみよう．

まずは学習データの準備．

In [ ]:
# 手書き数字データの入手
! wget -nc https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/minimnist.npz
rv = np.load('minimnist.npz')
datL = rv['datL'].astype(float)
labL = rv['labL']
datT = rv['datT'].astype(float)
labT = rv['labT']
print(datL.shape, labL.shape, datT.shape, labT.shape)

K = 10 # クラス数

# 学習データの用意
NL, D = datL.shape # 学習データの数と次元数
XL = datL/255
yL = labL

# テストデータの用意
NT, _ = datT.shape # テストデータの数
XT = datT/255
yT = labT

学習データは 5000 個．データの次元数は 784．

In [ ]:
# 事前確率を推定
py = np.empty(K)
for k in range(K):
    py[k] = np.sum(yL==k) / NL
print('### 事前確率')
print('p(y) = ', py)

2次元の例では，クラスごとに正規分布を当てはめる際に，平均と分散共分散行列を定義通り求めていたが，ここでは [sklearn.mixture.GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html) を用いることにする．
そうする理由は後述する．

GaussianMixture は，本来は混合正規分布（次回解説予定）のためのクラスであるが，`n_components=1` とすることで， 1つの正規分布を当てはめることができる（注）．
`covariance_type` や `reg_covar` というオプションの意味は後述する．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: 混合正規分布ではなく単一の正規分布を当てはめることが目的なら，
<a href="https://scikit-learn.org/stable/modules/covariance.html">https://scikit-learn.org/stable/modules/covariance.html</a>
で解説されているような手法を用いる方がより適切だが，ここでは分かりやすさおよび混合正規分布モデルの話との接続を考えて GaussianMixture を用いている．
</span>

In [ ]:
# p(x|y) を正規分布でモデル化（平均，共分散行列の推定）
mu = np.empty((K, D))
Gaussian = np.empty(K, dtype=object)
for k in range(K):
    XX = XL[labL==k, :]
    Gaussian[k] = GaussianMixture(n_components=1, covariance_type='full', reg_covar=1e-06)
    Gaussian[k].fit(XX)
    mu[k] = Gaussian[k].means_[0]

次のコードセルを実行すると，クラスごとの正規分布の平均を画像として可視化できる．

In [ ]:
# クラスごとの正規分布の平均を可視化
fig, ax = plt.subplots(1, K, figsize=(8, 2))
for k in range(K):
    img = mu[k].reshape((28, 28))
    ax[k].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[k].axis('off')
    ax[k].set_title(k)

fig.tight_layout()
plt.show()

学習データ，テストデータのそれぞれを識別させてみよう．

In [ ]:
### 学習データの識別

# 対数尤度を算出
LLL = np.empty([NL, K])
for k in range(K):
    # log p(y|x) = log p(y) + log p(x|y)
    LLL[:, k] = np.log(py[k]) + Gaussian[k].score_samples(XL)

# 尤度最大のクラスに識別
y_predict = np.argmax(LLL, axis=1)
ncL = np.sum(yL == y_predict)

### テストデータの識別

# 対数尤度を算出
LLT = np.empty([NT, K])
for k in range(K):
    # log p(y|x) = log p(y) + log p(x|y)
    LLT[:, k] = np.log(py[k]) + Gaussian[k].score_samples(XT)

# 尤度最大のクラスに識別
y_predict = np.argmax(LLT, axis=1)
ncT = np.sum(yT == y_predict)

print(f'学習: {ncL}/{NL} = {ncL/NL}')
print(f'テスト: {ncT}/{NT} = {ncT/NT}')

この問題の場合に分散共分散行列を定義通り計算するとうまくいかない理由は...

分散共分散行列の正則化と `reg_covar`

分散共分散行列の形を制約する話と `covariance_type`



---
### 判別分析や最短距離法との関係

---
## 混合正規分布モデル
---

---
### 単一の正規分布ではうまく表せそうにないデータ

The Old Faithful Geyser Dataset という2次元のデータ．
これは，米国イエローストーン国立公園内にある，「[The Old Faithful Geyser](https://ja.wikipedia.org/wiki/%E3%82%AA%E3%83%BC%E3%83%AB%E3%83%89%E3%83%BB%E3%83%95%E3%82%A7%E3%82%A4%E3%82%B9%E3%83%95%E3%83%AB%E3%83%BB%E3%82%AC%E3%82%A4%E3%82%B6%E3%83%BC)」
という間欠泉（一定周期で熱湯の噴出と停止を繰り返す）を観測して得られたデータ．

ここでは，以下の URL から取得したものを用いる．

https://gist.github.com/curran/4b59d1046d9e66f2787780ad51a1cd87




In [ ]:
URL = 'https://gist.github.com/curran/4b59d1046d9e66f2787780ad51a1cd87/raw/9ec906b78a98cf300947a37b56cfe70d01183200/data.tsv'
dfOFG = pd.read_csv(URL, sep='\t')
XOFG = dfOFG.to_numpy()
print(XOFG.shape)
dfOFG

eruptions は噴出の持続時間[分]，waiting は次の噴出までの時間[分]．
散布図は次の通り．

In [ ]:
# グラフを描く
fig, ax = plt.subplots()
ax.scatter(dfOFG['eruptions'], dfOFG['waiting'], s=5)
xmin, xmax = 0, 6
ymin, ymax = 40, 100
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel('Eruptions [minutes]')
ax.set_ylabel('Waiting [minutes]')
plt.show()

この2次元のデータに一つの正規分布を当てはめるのはよい考えとは言えなさそうである．
この例に限らず，世の中には単一の正規分布ではうまくモデル化できない．どうしよう？

---
### 混合正規分布モデル

**混合正規分布モデル** (Gaussian Mixture Model, GMM) は，複数の多次元正規分布を組み合わせて複雑な分布を表現するものである．

$K$ 個の正規分布 $\mathrm{N}(\pmb{x}; \pmb{\mu}_k, \Sigma_k)$ を用いて，個々のデータが次のようにして生成されるとする:

1. 確率 $w_k$ で $k$ 番目の正規分布が選ばれる． $0 \leq w_k \leq 1$ ($k = 1, 2, \ldots, K$) かつ $\sum_{k=1}^{K}w_k = 1$ とする．
1. その正規分布に従ってデータが生成される



GMM を式で表そう．
まず，どの正規分布を選ぶかを表す確率変数 $z$ を導入する．
$z$ は $1, 2, \ldots, K$ のいずれかをとる離散確率変数である．
すると，

$$
\begin{aligned}
p(z = k) &= w_k\\
p(\pmb{x}|z = k) &= \mathrm{N}(\pmb{x}; \pmb{\mu}_k, \Sigma_k)
\end{aligned}
$$

と表せる．このとき，

$$
\begin{aligned}
p(\pmb{x}) = \sum_{k=1}^{K} p(\pmb{x}|z = k) p(z = k)
\end{aligned}
$$

なので，$p(\pmb{x})$ は次式のように表される．

$$
\begin{aligned}
p(\pmb{x}) = \sum_{k=1}^{K} w_k \mathrm{N}(\pmb{x}; \pmb{\mu}_k, \Sigma_k)
\end{aligned}
$$

これが GMM の式である．
GMM のパラメータは，$w_k, \pmb{\mu}_k, \Sigma_k$ ($k = 1, 2, \ldots, K$) である．



上記の導出過程では， $z$ という確率変数が登場した．
確率変数 $\pmb{x}$ が観測できる（データが直接手に入る）のに対して，この確率変数 $z$ は観測できるものではない．
GMM において，データの生成過程に隠れて存在していると仮定した変数である．
このような変数のことを **潜在変数** (latent variable) という．
GMM は，潜在変数を持つ確率モデルの代表例である．


---
### 実験: Old Faithful Dataset に GMM を当てはめてみる

GMM のパラメータをどうやって推定するかという話は後回しにして， The Old Faithful Dataset Geyser Dataset に GMM を当てはめてみよう．
ここでは， [sklearn.mixture.GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html) を用いる．

In [ ]:
# GMM の当てはめ
K = 2
gmm = GaussianMixture(n_components=K)
gmm.fit(XOFG)

In [ ]:
# 推定されたモデルパラメータの表示
for k in range(K):
    print(f'### {k}-th component')
    print('weight = ', gmm.weights_[k])
    print('mu = ', gmm.means_[k])
    print('cov = ')
    print(gmm.covariances_[k])
    print()

In [ ]:
## データに正規分布を重ねて表示

# グラフ描画用のグリッドデータの作成
x_mesh, y_mesh = np.mgrid[xmin:xmax:(xmax-xmin)/100, ymin:ymax:(ymax-ymin)/100]
X_mesh = np.dstack((x_mesh, y_mesh))

# グラフ
fig, ax = plt.subplots()
ax.scatter(XOFG[:, 0], XOFG[:, 1], s=5)

# Gaussian を当てはめた結果
for k in range(K):
    mu = gmm.means_[k]
    cov = gmm.covariances_[k]
    ax.contour(x_mesh, y_mesh, multivariate_normal.pdf(X_mesh, mean=mu, cov=cov))
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

データの生成過程をモデル化しているので，疑似乱数を使ってこの GMM が表す分布に従うサンプルを無限に作り出すことができる．

In [ ]:
# データを生成
N = 200
z = rng.choice(K, size=N, p=gmm.weights_)
print(z)

Xgen = np.empty((N, 2))
for k in range(K):
    Nk = np.sum(z == k)
    mu = gmm.means_[k]
    cov = gmm.covariances_[k]
    Xgen[z == k, :] = rng.multivariate_normal(mu, cov, size=Nk)

# 最初の10個
for n in range(10):
    print(z[n], Xgen[n, :])

In [ ]:
# グラフを描く
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
ax[0].scatter(XOFG[:, 0], XOFG[:, 1], label='observed data', s=5)
ax[0].scatter(Xgen[:, 0], Xgen[:, 1], label='generated', s=5)
ax[0].set_xlim(xmin, xmax)
ax[0].set_ylim(ymin, ymax)
ax[0].legend()
ax[1].scatter(Xgen[z == 0, 0], Xgen[z == 0, 1], label='k = 0', color='g', s=5)
ax[1].scatter(Xgen[z == 1, 0], Xgen[z == 1, 1], label='k = 1', color='m', s=5)
ax[1].set_xlim(xmin, xmax)
ax[1].set_ylim(ymin, ymax)
ax[1].legend()
plt.show()

左は，GMMを使って生成したデータを元のデータと重ねて描いたもの．
右は，その生成されたデータがどちらの正規分布から生成されたのかを色分けして描いたもの．